## Module 2: Vector Search

This module extends the RAG pipeline from the first module with vector search. Vector search matches documents by semantic meaning instead of exact keyword overlap. Traditional keyword search matches exact words, whereas vector search utilizes a neural embedding model to convert text into numerical arrays (vectors) that capture the actual semantic meaning of a query. This process operates in two stages: 

1. Offline indexing phase that stores document vectors
2. Online querying phase that translates a user's question to find the closest matches based on a distance metric like cosine similarity 

While vector search is superior for natural language and paraphrasing, it introduces significant operational complexity and cost; therefore, the best approach is to start with text search and eventually combine both methods into a hybrid search for optimal performance.

In [1]:
# make sure .env file is loaded
from dotenv import load_dotenv
print(load_dotenv())

# check whether the OpenAI client works
from openai import OpenAI
openai_client = OpenAI()

True


### Section 1: Embeddings

Turning text into vectors called *embedding* and the obtained vectors are called *embeddings*. The idea comes from [word2vec](https://en.wikipedia.org/wiki/Word2vec). The model learns to place words as points in a multi-dimensional space. Words with similar meanings land close to each other. The same idea works for entire sentences. An embedding model gets text input and returns a fixed-length array of numbers. It is trained so that texts with similar meanings get similar vectors. Here, [sentence-transformers](https://www.sbert.net/) are used, which is a popular open-source library for embeddings. It runs locally without API costs. To install:

```python
uv add sentence-transformers
```

Sentence-transformers supports many models. The right one depends on the task, the language, and the resources. It is better to try a few on models on the data and pick the best working one. Larger models are usually slower, so for a small task, a smalles model should be preferred.

In [2]:
from sentence_transformers import SentenceTransformer
# load a pretrained sentence transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
# sample question embedded
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
print(f"Shape of q1 embedding: {v1.shape}")

# sample document is embedded
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
print(f"Shape of d embedding: {dv.shape}")

# compare the sample question against the document with dot product
print(f"Cosine similarity of q1 and the document: {v1.dot(dv)}")

Shape of q1 embedding: (384,)
Shape of d embedding: (384,)
Cosine similarity of q1 and the document: 0.32332396507263184


In [13]:
# a totally unrelated sample question embedded
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
print(f"Shape of q2 embedding: {v2.shape}")

# compare the second sample question against the document with dot product
print(f"Cosine similarity of q2 and the document: {v2.dot(dv)}")

Shape of q2 embedding: (384,)
Cosine similarity of q2 and the document: 0.019730573520064354


Two vectors with similar values point to texts about similar things. The score is ear 0 which means the vectors are orthogonal, hence totally unrelated. The initial question has a higher similarity. 

**Cosine similarity:**
The selected all-MiniLM-L6-v2 model outputs normalized vectors. Hence, the dot product equals to the cosine similarity. Cosine similarity measures the angle between two vectors, ignoring their length:

- 1.0 = same direction (similar)
- 0.0 = perpendicular (unrelated)
- -1.0 = opposite direction (opposite meaning)

Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):

- cos(0) = 1 - vectors point in the same direction
- cos(90) = 0 - vectors are perpendicular
- cos(180) = -1 - vectors point in opposite directions

In practice, getting a cosine similarity below 0 is very rare. The embedding model maps text to a region of the vector space where most vectors have positive components. There is no concept of "opposite meaning" that maps to a vector pointing the other way.

### Section 2: Embedding the Dataset

This section loads the same FAQ data from the first module and apply embedding to the whole dataset. Each document in the FAQ dataset is a Python dictionary with a question and an answer. To match a query against the question text and the answer text in the index, both the question and the answer will be embedded together.

In [18]:
from ingest import load_faq_data

# load FAQ data
documents = load_faq_data()

# check a sample
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [19]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

# total number of texts
print(f"Total texts: {len(texts)}")

# check a sample
texts[10]

Total texts: 1350


'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [21]:
# import tqdm to watch the progress (uv add tqdm)
from tqdm.auto import tqdm

batch_size = 50
vectors = []

# chunk the dataset into batches of 50 and encode each batch
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

# total number of vectors
print(f"Total vectors: {len(vectors)}")

  0%|          | 0/27 [00:00<?, ?it/s]

Total vectors: 1350


In [22]:
import numpy as np

# turn vectors into a matrix where rows are documents (vectors), and cols are dimensions of them
X = np.array(vectors)

print(f"There are {X.shape[0]} vectors with {X.shape[1]} dimensions.")

There are 1350 vectors with 384 dimensions.


### Section 3: Vector Search

The matrix `X` contains all the document embeddings. Rather than checking the cosine similarity of a question with each document with a vector-vector multiplication, it is much simpler and faster to use matrices and perform a matrix-vector multiplication.

In [24]:
# define a sample question and its embedding
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

# get scores of all documents for the current question at hand
scores = X.dot(v_query)

# do not use loops for it, it is much slower
# scores = [v_query.dot(X[i]) for i in range(len(X))]

# get the highest scored document, which is the most similar
idx = np.argmax(scores)
print(f"The most similar document index: {idx}, score: {scores[idx]}")
documents[idx]

The most similar document index: 2, score: 0.7629410028457642


{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

The answer to a question can be spread across several documents. One holds part of it, another fills in the rest. Sometimes the top result is not the right one but the second is. Hence all 5 top results are sent to the LLM to be combined. The number can be later arranged by evaluating the search quality.

In [ ]:
# return top 5 scores (argsort sorts from lowest to highest)
top5 = np.argsort(scores)[-5:]

# sort top5 scores from highest to lowest
top5 = top5[::-1]

# or as a one line turning min-sort to max-sort: top5 = np.argsort(-scores)[:5]

# show the best 5 results
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

### Section 4: Vector Search with minsearch

This section handles the vector search using the minsearch library. Rather than performing all the tasks of embeddings, computing products, finding and sorting the best matches, an API will be used instead to perform the vector search.